# Calibration and Recalibration Stress Test

This notebook is Milestone 08 for the revised fraud-detection paper.

Research purpose:

- Test whether strict fraud-detection model scores are well calibrated.
- Compare raw probabilities with validation-set recalibration.
- Quantify whether Platt sigmoid and isotonic recalibration improve Brier score, log loss, Expected Calibration Error, and operating cost.
- Save every checkpoint, result table, calibration curve, log, and 600-DPI figure to Google Drive.

This is a probability-quality experiment. It does not add new features and does not change the leakage-safe protocol.

In [ ]:
!pip -q install -U imbalanced-learn lightgbm xgboost seaborn
!pip -q install -q "pyarrow>=19,<24"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Drive screenshot showed: My Drive > Credit__ > Datasets
PROJECT_ROOT = Path('/content/drive/MyDrive/Credit__')

DATASET_SEARCH_DIRS = [
    PROJECT_ROOT / 'Datasets',
    PROJECT_ROOT / '00_source_datasets',
    PROJECT_ROOT,
]

RUN_NAME = 'calibration_recalibration_v1'
EXPERIMENT_ROOT = PROJECT_ROOT / '08_calibration_recalibration' / RUN_NAME
CHECKPOINT_DIR = EXPERIMENT_ROOT / 'checkpoints'
BIN_CHECKPOINT_DIR = EXPERIMENT_ROOT / 'calibration_bins'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
FIGURES_DIR = EXPERIMENT_ROOT / 'figures'
LOGS_DIR = EXPERIMENT_ROOT / 'logs'

for d in [EXPERIMENT_ROOT, CHECKPOINT_DIR, BIN_CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 101, 202, 303, 404]
MODEL_NAMES = ['xgboost', 'lightgbm']
CALIBRATORS = ['RAW', 'PLATT_SIGMOID', 'ISOTONIC']

TEST_SIZE = 0.20
CALIBRATION_SIZE = 0.20
N_CALIBRATION_BINS = 10

CAPS = {
    'D1_creditcard': None,
    'D2_fraudTest': None,
    'D3_PaySim': 1_000_000,
}

SMOKE_TEST = False
SMOKE_CAP = 50_000

print('Project root:', PROJECT_ROOT)
print('Experiment root:', EXPERIMENT_ROOT)
print('Seeds:', SEEDS)

In [ ]:
import json
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except Exception as exc:
    xgb = None
    print('XGBoost unavailable:', exc)

try:
    import lightgbm as lgb
except Exception as exc:
    lgb = None
    print('LightGBM unavailable:', exc)

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'legend.title_fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

def now():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def write_csv_atomic(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def append_log(message):
    line = f'[{now()}] {message}'
    print(line)
    with open(LOGS_DIR / 'run.log', 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def checkpoint_path(seed, dataset, model_name):
    return CHECKPOINT_DIR / f'seed{seed}__{dataset}__{model_name}.csv'

def bin_checkpoint_path(seed, dataset, model_name):
    return BIN_CHECKPOINT_DIR / f'bins__seed{seed}__{dataset}__{model_name}.csv'

def is_done(seed, dataset, model_name):
    return checkpoint_path(seed, dataset, model_name).exists() and bin_checkpoint_path(seed, dataset, model_name).exists()

In [ ]:
def find_dataset_file(file_name):
    candidates = []
    for directory in DATASET_SEARCH_DIRS:
        candidates.append(directory / file_name)
        if directory.exists():
            candidates.extend(directory.glob(f'**/{file_name}'))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find {file_name}')

DATASET_FILES = {
    'D1_creditcard': find_dataset_file('creditcard.csv'),
    'D2_fraudTest': find_dataset_file('fraudTest.csv'),
    'D3_PaySim': find_dataset_file('PS.csv'),
}

for name, path in DATASET_FILES.items():
    append_log(f'{name}: {path}')

write_json(EXPERIMENT_ROOT / 'dataset_file_manifest.json', {k: str(v) for k, v in DATASET_FILES.items()})

In [ ]:
def temporal_subsample(df, order_col, cap):
    if SMOKE_TEST:
        cap = min(SMOKE_CAP, len(df))
    if cap is None or len(df) <= cap:
        return df.reset_index(drop=True)
    positions = np.linspace(0, len(df) - 1, num=cap, dtype=np.int64)
    return df.iloc[positions].reset_index(drop=True)

def load_dataset(dataset_name):
    path = DATASET_FILES[dataset_name]
    df = pd.read_csv(path, low_memory=False)

    if dataset_name == 'D1_creditcard':
        df = df.drop_duplicates().reset_index(drop=True)
        label_col = 'Class'
        df['__order__'] = pd.to_numeric(df['Time'], errors='coerce')

    elif dataset_name == 'D2_fraudTest':
        label_col = 'is_fraud'
        df['__order__'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')

    elif dataset_name == 'D3_PaySim':
        label_col = 'isFraud'
        df = df.dropna(subset=[label_col]).reset_index(drop=True)
        df[label_col] = df[label_col].astype(int)
        df['__order__'] = pd.to_numeric(df['step'], errors='coerce')

    else:
        raise ValueError(dataset_name)

    df = df.dropna(subset=[label_col]).sort_values('__order__').reset_index(drop=True)
    df = temporal_subsample(df, '__order__', CAPS[dataset_name])
    fraud = int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum())
    append_log(f'{dataset_name}: rows={len(df):,}, fraud={fraud:,}, rate={fraud/len(df)*100:.4f}%')
    return df, label_col

def prepare_features(df, dataset_name):
    df = df.copy()

    if dataset_name == 'D1_creditcard':
        y = df['Class'].astype(int)
        X = df[[f'V{i}' for i in range(1, 29)] + ['Amount']].copy()
        return X, y

    if dataset_name == 'D2_fraudTest':
        df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')
        df['dob_dt'] = pd.to_datetime(df['dob'], errors='coerce')
        df['trans_hour'] = df['trans_dt'].dt.hour
        df['trans_dayofweek'] = df['trans_dt'].dt.dayofweek
        df['trans_month'] = df['trans_dt'].dt.month
        df['age_years'] = (df['trans_dt'] - df['dob_dt']).dt.days / 365.25
        keep = ['category', 'amt', 'state', 'city_pop', 'trans_hour', 'trans_dayofweek', 'trans_month', 'age_years']
        return df[keep].copy(), df['is_fraud'].astype(int)

    if dataset_name == 'D3_PaySim':
        keep = ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']
        return df[keep].copy(), df['isFraud'].astype(int)

    raise ValueError(dataset_name)

def make_preprocessor(X):
    numeric_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler(with_centering=False)),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=True)),
    ])
    return ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])

In [ ]:
def build_model(model_name, seed, scale_pos_weight=1.0):
    if model_name == 'xgboost':
        if xgb is None:
            raise RuntimeError('XGBoost not available')
        return xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            objective='binary:logistic',
            eval_metric='aucpr',
            tree_method='hist',
            device='cuda',
            scale_pos_weight=scale_pos_weight,
            random_state=seed,
            n_jobs=-1,
        )
    if model_name == 'lightgbm':
        if lgb is None:
            raise RuntimeError('LightGBM not available')
        return lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.04,
            num_leaves=63,
            class_weight='balanced',
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )
    raise ValueError(model_name)

def get_scores(model, X):
    return model.predict_proba(X)[:, 1]

def clip_prob(scores):
    return np.clip(np.asarray(scores, dtype=float), 1e-6, 1 - 1e-6)

def expected_calibration_error(y_true, scores, n_bins=N_CALIBRATION_BINS):
    y_true = np.asarray(y_true)
    scores = clip_prob(scores)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        if i == n_bins - 1:
            mask = (scores >= lo) & (scores <= hi)
        else:
            mask = (scores >= lo) & (scores < hi)
        n = int(mask.sum())
        if n == 0:
            rows.append({'bin': i, 'bin_low': lo, 'bin_high': hi, 'n': 0, 'mean_score': np.nan, 'observed_rate': np.nan, 'abs_gap': np.nan})
            continue
        mean_score = float(scores[mask].mean())
        observed_rate = float(y_true[mask].mean())
        gap = abs(observed_rate - mean_score)
        ece += (n / len(y_true)) * gap
        mce = max(mce, gap)
        rows.append({'bin': i, 'bin_low': lo, 'bin_high': hi, 'n': n, 'mean_score': mean_score, 'observed_rate': observed_rate, 'abs_gap': gap})
    return float(ece), float(mce), pd.DataFrame(rows)

def choose_threshold(y_val, scores_val, c_fn=100, c_fp=1):
    best = None
    for tau in np.round(np.arange(0.01, 0.91, 0.01), 2):
        pred = (scores_val >= tau).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
        cost = c_fp * fp + c_fn * fn
        mcc = matthews_corrcoef(y_val, pred) if len(np.unique(pred)) > 1 else 0.0
        candidate = (cost, fn, fp, -mcc, tau)
        if best is None or candidate < best:
            best = candidate
    return float(best[-1])

def evaluate_predictions(y_true, scores, tau):
    scores = clip_prob(scores)
    pred = (scores >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    n = len(y_true)
    ece, mce, bins = expected_calibration_error(y_true, scores)
    return {
        'threshold': float(tau),
        'n_test': int(n),
        'test_fraud': int(np.sum(y_true == 1)),
        'test_prevalence': float(np.mean(y_true == 1)),
        'tp': int(tp),
        'fp': int(fp),
        'tn': int(tn),
        'fn': int(fn),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, pred)) if len(np.unique(pred)) > 1 else 0.0,
        'average_precision': float(average_precision_score(y_true, scores)),
        'roc_auc': float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': float(brier_score_loss(y_true, scores)),
        'log_loss': float(log_loss(y_true, scores, labels=[0, 1])),
        'ece': ece,
        'mce': mce,
        'mean_score': float(np.mean(scores)),
        'false_alerts_per_10k': float(fp / n * 10_000),
        'missed_frauds_per_10k': float(fn / n * 10_000),
        'cost_per_10k': float((fp + 100 * fn) / n * 10_000),
    }, bins

In [ ]:
def split_train_calibration_test(X, y, seed):
    idx = np.arange(len(y))
    train_cal_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, stratify=y, random_state=seed)
    y_train_cal = y.iloc[train_cal_idx]
    cal_relative = CALIBRATION_SIZE / (1.0 - TEST_SIZE)
    train_idx, cal_idx = train_test_split(train_cal_idx, test_size=cal_relative, stratify=y_train_cal, random_state=seed)
    return np.array(train_idx), np.array(cal_idx), np.array(test_idx)

def fit_recalibrators(y_cal, cal_raw_scores):
    platt = LogisticRegression(max_iter=1000, solver='lbfgs')
    platt.fit(clip_prob(cal_raw_scores).reshape(-1, 1), y_cal)

    isotonic = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
    isotonic.fit(clip_prob(cal_raw_scores), y_cal)
    return platt, isotonic

def apply_calibrator(calibrator_name, scores, platt, isotonic):
    scores = clip_prob(scores)
    if calibrator_name == 'RAW':
        return scores
    if calibrator_name == 'PLATT_SIGMOID':
        return clip_prob(platt.predict_proba(scores.reshape(-1, 1))[:, 1])
    if calibrator_name == 'ISOTONIC':
        return clip_prob(isotonic.predict(scores))
    raise ValueError(calibrator_name)

def run_calibration_pipeline(X, y, model_name, seed):
    train_idx, cal_idx, test_idx = split_train_calibration_test(X, y, seed)
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_cal, y_cal = X.iloc[cal_idx], y.iloc[cal_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    preprocessor = make_preprocessor(X_train)
    X_train_t = preprocessor.fit_transform(X_train)
    X_cal_t = preprocessor.transform(X_cal)
    X_test_t = preprocessor.transform(X_test)

    pos = max(int(np.sum(y_train == 1)), 1)
    neg = max(int(np.sum(y_train == 0)), 1)
    model = build_model(model_name, seed, scale_pos_weight=neg / pos)
    t0 = time.time()
    model.fit(X_train_t, y_train)
    fit_seconds = time.time() - t0

    cal_raw_scores = get_scores(model, X_cal_t)
    test_raw_scores = get_scores(model, X_test_t)
    platt, isotonic = fit_recalibrators(y_cal, cal_raw_scores)

    metric_rows = []
    bin_rows = []
    for calibrator_name in CALIBRATORS:
        cal_scores = apply_calibrator(calibrator_name, cal_raw_scores, platt, isotonic)
        test_scores = apply_calibrator(calibrator_name, test_raw_scores, platt, isotonic)
        threshold = choose_threshold(y_cal, cal_scores)
        metrics, bins = evaluate_predictions(y_test.reset_index(drop=True), test_scores, threshold)
        row = {
            'calibrator': calibrator_name,
            'fit_seconds': float(fit_seconds),
            'train_n': int(len(y_train)),
            'train_fraud': int(np.sum(y_train == 1)),
            'calibration_n': int(len(y_cal)),
            'calibration_fraud': int(np.sum(y_cal == 1)),
            **metrics,
        }
        metric_rows.append(row)
        bins = bins.copy()
        bins['calibrator'] = calibrator_name
        bin_rows.append(bins)
    return pd.DataFrame(metric_rows), pd.concat(bin_rows, ignore_index=True)

In [ ]:
def run_one(seed, dataset_name, model_name):
    if is_done(seed, dataset_name, model_name):
        append_log(f'SKIP seed={seed} {dataset_name} / {model_name}')
        return (
            pd.read_csv(checkpoint_path(seed, dataset_name, model_name)),
            pd.read_csv(bin_checkpoint_path(seed, dataset_name, model_name)),
        )

    df, label_col = load_dataset(dataset_name)
    X, y = prepare_features(df, dataset_name)
    metrics, bins = run_calibration_pipeline(X, y, model_name, seed)

    common = {
        'run_name': RUN_NAME,
        'timestamp': now(),
        'seed': seed,
        'dataset': dataset_name,
        'model': model_name,
        'feature_policy': 'strict',
        'smoke_test': SMOKE_TEST,
        'rows_used': int(len(df)),
        'fraud_rows_used': int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum()),
        'cap_used': CAPS[dataset_name],
    }
    for key, value in common.items():
        metrics[key] = value
        bins[key] = value

    ordered_cols = list(common.keys()) + [c for c in metrics.columns if c not in common]
    metrics = metrics[ordered_cols]
    bin_ordered_cols = list(common.keys()) + [c for c in bins.columns if c not in common]
    bins = bins[bin_ordered_cols]

    write_csv_atomic(checkpoint_path(seed, dataset_name, model_name), metrics)
    write_csv_atomic(bin_checkpoint_path(seed, dataset_name, model_name), bins)
    raw = metrics[metrics['calibrator'] == 'RAW'].iloc[0]
    append_log(
        f'DONE seed={seed} {dataset_name} / {model_name}: '
        f'RAW Brier={raw["brier"]:.5f}, ECE={raw["ece"]:.5f}, cost={raw["cost_per_10k"]:.2f}'
    )
    return metrics, bins

all_metric_frames = []
all_bin_frames = []
for seed in SEEDS:
    for dataset_name in ['D1_creditcard', 'D2_fraudTest', 'D3_PaySim']:
        for model_name in MODEL_NAMES:
            try:
                metrics, bins = run_one(seed, dataset_name, model_name)
                all_metric_frames.append(metrics)
                all_bin_frames.append(bins)
                write_csv_atomic(RESULTS_DIR / 'partial_results_latest.csv', pd.concat(all_metric_frames, ignore_index=True))
                write_csv_atomic(RESULTS_DIR / 'partial_calibration_bins_latest.csv', pd.concat(all_bin_frames, ignore_index=True))
            except Exception as exc:
                append_log(f'ERROR seed={seed} {dataset_name} / {model_name}: {repr(exc)}')
                write_json(CHECKPOINT_DIR / f'ERROR__seed{seed}__{dataset_name}__{model_name}.json', {
                    'seed': seed,
                    'dataset': dataset_name,
                    'model': model_name,
                    'error': repr(exc),
                    'timestamp': now(),
                })

metric_frames = [pd.read_csv(p) for p in sorted(CHECKPOINT_DIR.glob('seed*.csv'))]
bin_frames = [pd.read_csv(p) for p in sorted(BIN_CHECKPOINT_DIR.glob('bins__seed*.csv'))]
results = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
bin_results = pd.concat(bin_frames, ignore_index=True) if bin_frames else pd.DataFrame()
write_csv_atomic(RESULTS_DIR / 'calibration_recalibration_results.csv', results)
write_csv_atomic(RESULTS_DIR / 'calibration_curve_bins.csv', bin_results)
results

In [ ]:
results = pd.read_csv(RESULTS_DIR / 'calibration_recalibration_results.csv')

raw = results[results['calibrator'] == 'RAW'][[
    'seed', 'dataset', 'model',
    'brier', 'ece', 'mce', 'log_loss', 'cost_per_10k', 'mcc', 'precision', 'recall'
]].rename(columns={
    'brier': 'raw_brier',
    'ece': 'raw_ece',
    'mce': 'raw_mce',
    'log_loss': 'raw_log_loss',
    'cost_per_10k': 'raw_cost_per_10k',
    'mcc': 'raw_mcc',
    'precision': 'raw_precision',
    'recall': 'raw_recall',
})

index = results.merge(raw, on=['seed', 'dataset', 'model'], how='left')
index['brier_reduction'] = index['raw_brier'] - index['brier']
index['ece_reduction'] = index['raw_ece'] - index['ece']
index['mce_reduction'] = index['raw_mce'] - index['mce']
index['log_loss_reduction'] = index['raw_log_loss'] - index['log_loss']
index['cost_delta_per_10k'] = index['cost_per_10k'] - index['raw_cost_per_10k']
index['mcc_delta'] = index['mcc'] - index['raw_mcc']
index['precision_delta'] = index['precision'] - index['raw_precision']
index['recall_delta'] = index['recall'] - index['raw_recall']

write_csv_atomic(RESULTS_DIR / 'calibration_improvement_index.csv', index)

summary = (
    index
    .groupby(['dataset', 'model', 'calibrator'])
    .agg(
        seeds=('seed', 'nunique'),
        brier_mean=('brier', 'mean'),
        brier_std=('brier', 'std'),
        brier_reduction_mean=('brier_reduction', 'mean'),
        ece_mean=('ece', 'mean'),
        ece_std=('ece', 'std'),
        ece_reduction_mean=('ece_reduction', 'mean'),
        log_loss_mean=('log_loss', 'mean'),
        log_loss_reduction_mean=('log_loss_reduction', 'mean'),
        cost_mean=('cost_per_10k', 'mean'),
        cost_delta_mean=('cost_delta_per_10k', 'mean'),
        mcc_mean=('mcc', 'mean'),
        mcc_delta_mean=('mcc_delta', 'mean'),
    )
    .reset_index()
)
write_csv_atomic(RESULTS_DIR / 'calibration_recalibration_summary.csv', summary)
summary

In [ ]:
index = pd.read_csv(RESULTS_DIR / 'calibration_improvement_index.csv')
summary = pd.read_csv(RESULTS_DIR / 'calibration_recalibration_summary.csv')
bins = pd.read_csv(RESULTS_DIR / 'calibration_curve_bins.csv')

dataset_labels = {'D1_creditcard': 'D1', 'D2_fraudTest': 'D2', 'D3_PaySim': 'D3'}
model_labels = {'xgboost': 'XGBoost', 'lightgbm': 'LightGBM'}
calibrator_labels = {'RAW': 'Raw', 'PLATT_SIGMOID': 'Platt', 'ISOTONIC': 'Isotonic'}

plot_df = index[index['calibrator'] != 'RAW'].copy()
plot_df['dataset_label'] = plot_df['dataset'].map(dataset_labels)
plot_df['model_label'] = plot_df['model'].map(model_labels)
plot_df['calibrator_label'] = plot_df['calibrator'].map(calibrator_labels)
plot_df['method_label'] = plot_df['model_label'] + ' / ' + plot_df['calibrator_label']

for metric, ylabel, filename in [
    ('brier_reduction', 'Brier score reduction vs raw', 'calibration_brier_reduction_by_dataset'),
    ('ece_reduction', 'ECE reduction vs raw', 'calibration_ece_reduction_by_dataset'),
    ('cost_delta_per_10k', 'Cost delta per 10,000 vs raw', 'calibration_cost_delta_by_dataset'),
]:
    fig, ax = plt.subplots(figsize=(8.0, 4.4))
    sns.barplot(
        data=plot_df,
        x='dataset_label',
        y=metric,
        hue='method_label',
        errorbar='sd',
        edgecolor='black',
        linewidth=0.45,
        ax=ax,
    )
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Dataset')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' across five seeds')
    ax.legend(title='Model / recalibrator', loc='best', frameon=True, ncol=2)
    ax.grid(axis='y', linestyle='--', linewidth=0.45, alpha=0.55)
    ax.grid(axis='x', visible=False)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    png = FIGURES_DIR / f'{filename}.png'
    pdf = FIGURES_DIR / f'{filename}.pdf'
    fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.show()
    append_log(f'Saved figure: {png}')

curve = bins[bins['n'] > 0].copy()
curve['dataset_label'] = curve['dataset'].map(dataset_labels)
curve['calibrator_label'] = curve['calibrator'].map(calibrator_labels)
curve_summary = (
    curve
    .groupby(['dataset_label', 'calibrator_label', 'bin'])
    .agg(mean_score=('mean_score', 'mean'), observed_rate=('observed_rate', 'mean'))
    .reset_index()
)

for dataset_label in ['D1', 'D2', 'D3']:
    fig, ax = plt.subplots(figsize=(5.0, 4.4))
    part = curve_summary[curve_summary['dataset_label'] == dataset_label]
    sns.lineplot(
        data=part,
        x='mean_score',
        y='observed_rate',
        hue='calibrator_label',
        marker='o',
        linewidth=1.6,
        ax=ax,
    )
    ax.plot([0, 1], [0, 1], color='black', linewidth=0.9, linestyle='--')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed fraud rate')
    ax.set_title(f'Calibration curve: {dataset_label}')
    ax.legend(title='Probability', loc='best', frameon=True)
    ax.grid(axis='both', linestyle='--', linewidth=0.45, alpha=0.55)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    png = FIGURES_DIR / f'calibration_curve_{dataset_label}.png'
    pdf = FIGURES_DIR / f'calibration_curve_{dataset_label}.pdf'
    fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.show()
    append_log(f'Saved figure: {png}')

In [ ]:
results = pd.read_csv(RESULTS_DIR / 'calibration_recalibration_results.csv')
index = pd.read_csv(RESULTS_DIR / 'calibration_improvement_index.csv')
summary = pd.read_csv(RESULTS_DIR / 'calibration_recalibration_summary.csv')
bins = pd.read_csv(RESULTS_DIR / 'calibration_curve_bins.csv')

run_summary = {
    'run_name': RUN_NAME,
    'completed_at': now(),
    'project_root': str(PROJECT_ROOT),
    'experiment_root': str(EXPERIMENT_ROOT),
    'seeds': SEEDS,
    'models': MODEL_NAMES,
    'calibrators': CALIBRATORS,
    'n_calibration_bins': N_CALIBRATION_BINS,
    'caps': CAPS,
    'results_file': str(RESULTS_DIR / 'calibration_recalibration_results.csv'),
    'index_file': str(RESULTS_DIR / 'calibration_improvement_index.csv'),
    'summary_file': str(RESULTS_DIR / 'calibration_recalibration_summary.csv'),
    'curve_bins_file': str(RESULTS_DIR / 'calibration_curve_bins.csv'),
    'figures_dir': str(FIGURES_DIR),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'n_result_rows': int(len(results)),
    'n_index_rows': int(len(index)),
    'n_summary_rows': int(len(summary)),
    'n_curve_bin_rows': int(len(bins)),
}
write_json(EXPERIMENT_ROOT / 'run_summary.json', run_summary)
run_summary

## Expected Runtime on Colab T4

Expected runtime is approximately **35-75 minutes**.

Why:

- Only 30 model fits are required: 3 datasets x 2 models x 5 seeds.
- Platt sigmoid and isotonic recalibration are lightweight compared with model training.
- D3 PaySim is capped to 1,000,000 timeline-sampled rows by default.

If Colab disconnects, rerun from the top. Existing checkpoint CSVs and calibration-bin checkpoint CSVs are skipped.